<a href="https://colab.research.google.com/github/vereesmort/pykeen-colab/blob/main/Reproduce_Refactor_TFDecagon_PyKEEN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount drive

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [13]:
%cd /content/drive//MyDrive/Colab_Projects

/content/drive/MyDrive/Colab_Projects


## Git

In [14]:
!git clone "https://github.com/vereesmort/TF_Decagon_Static.git"

Cloning into 'TF_Decagon_Static'...
remote: Enumerating objects: 1481, done.
remote: Counting objects: 100% (1481/1481), done.
remote: Compressing objects: 100% (811/811), done.
remote: Total 1481 (delta 98), reused 1438 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (1481/1481), 18.80 MiB | 13.80 MiB/s, done.
Resolving deltas: 100% (98/98), done.
Updating files: 100% (729/729), done.


In [15]:
%cd /content/drive//MyDrive/Colab_Projects/TF_Decagon_Static

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static


In [7]:
!git status

^C


In [8]:
!rm -f .git/index.lock
!rm -f .git/refs/heads/main.lock


In [9]:
!git pull "https://github.com/vereesmort/TF_Decagon_Static.git" --verbose

POST git-upload-pack (102 bytes)
From https://github.com/vereesmort/TF_Decagon_Static
 * branch            HEAD       -> FETCH_HEAD
Updating b379f6a..ecfdfc7
^C


# Download raw dataset

In [ ]:
%cd /content/drive//MyDrive/Colab_Projects/TF_Decagon_Static/data/raw/

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/raw


In [ ]:
!bash download_and_unpack.sh

--2026-05-05 15:53:47--  http://snap.stanford.edu/decagon/bio-decagon-ppi.tar.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2396721 (2.3M) [application/x-gzip]
Saving to: ‘bio-decagon-ppi.tar.gz’

bio-decagon-ppi.tar 100%[===================>]   2.29M  5.74MB/s    in 0.4s    

2026-05-05 15:53:47 (5.74 MB/s) - ‘bio-decagon-ppi.tar.gz’ saved [2396721/2396721]

--2026-05-05 15:53:47--  http://snap.stanford.edu/decagon/bio-decagon-targets.tar.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 61835 (60K) [application/x-gzip]
Saving to: ‘bio-decagon-targets.tar.gz’

bio-decagon-targets 100%[===================>]  60.39K  --.-KB/s    in 0.04s   

2026-05-05 15:53:47 (1.48 MB/s) - ‘bi

## Stage 2 — Process raw data

In [ ]:
!wc -l bio-decagon-combo.csv   # expect 4,651,132 (including header)
!wc -l bio-decagon-ppi.csv     # expect 715,613

wc: bio-decagon-combo.csv: No such file or directory
wc: bio-decagon-ppi.csv: No such file or directory


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/
!python process_raw_data.py \
  --raw_dir /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/raw/ \
  --out_dir /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/

/content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/processed/polypharmacy/
!wc -l polypharmacy_edges.tsv  # 4,576,287

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/processed/polypharmacy
4576287 polypharmacy_edges.tsv


## Stage 3 — Create the holdout split

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/
!python processed/polypharmacy/split_by_polypharmacy_side_effect.py \
  --edgelist /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/polypharmacy/polypharmacy_edges.tsv \
  --combo_csv /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/raw/bio-decagon-combo.csv \
  --out_dir /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/polypharmacy

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data
Found total overlap of holdout nodes with train nodes. Saving..
Wrote train/holdout under /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/polypharmacy


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/polypharmacy
!wc -l train_polypharmacy.tsv    # 4,118,226
!wc -l holdout_polypharmacy.tsv  # 458,061

/content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/processed/polypharmacy
4118226 train_polypharmacy.tsv
458061 holdout_polypharmacy.tsv


## Stage 4 — Build the Selfloops graph
The Selfloops graph encodes monopharmacy SEs as self-loop edges (drug → SE_type → same_drug) rather than node feature vectors.

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/graphs/selfloops/
!python process_selfloop_graph.py

/content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/graphs/selfloops


## Stage 5 — Prepare the PyKEEN dataset
Splits `edgelist_selfloops.tsv` 80/10/10

In [16]:
!pip install pykeen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/graphs/

/content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/graphs


In [ ]:
!python make_pykeen_datasets.py \
  --edgelist selfloops/edgelist_selfloops.tsv \
  --out_dir  pykeen_factories/ \
  --seed     0

Loading edgelist...
  5,027,505 total edges
  train: 4,022,003  valid: 502,751  test: 502,751
Building TriplesFactory from full vocabulary...
  19,734 entities  |  11,149 relations
  Saved pykeen_factories/train_tf.pt
  Saved pykeen_factories/valid_tf.pt
  Saved pykeen_factories/test_tf.pt
  Saved entity_to_id.json and relation_to_id.json

Done. Use --dataset_dir with train_simplE_pykeen.py.


## Stage 6 — Train SimplE

In [17]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("torch CUDA version (build):", torch.version.cuda)

torch: 2.10.0+cu128
cuda available: True
device count: 1
device name: Tesla T4
torch CUDA version (build): 12.8


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static


In [ ]:
!python training/train_simplE_pykeen.py \
        --dataset_dir data/graphs/pykeen_factories \
        --out_dir     results/simplE_selfloops \
        --max_epochs  2 \
        --seed        42

Device: cuda  |  seed: 42
Loading datasets...
  train: 4,022,003  valid: 502,751  test:  502,751
  entities: 19,734  relations: 11,149
  Entity init: xavier_normal_ (selfloops mode)
No random seed is specified. This may lead to non-reproducible results.
  Parameters: 15,812,096

Starting training (max 2 epochs)...
  embedding_dim=256  lr=0.0107609491076982  batch_size=256
  entity_dropout=0.0679910727776587  relation_dropout=0.1253491407260298
  Early stopping: patience=10 checks × 5 epochs = 50 epochs max wait
Training epochs on cuda:0:   0% 0/2 [00:00<?, ?epoch/s]
Training batches on cuda:0:   0% 0.00/1.60k [00:00<?, ?batch/s]
Training batches on cuda:0:   0% 1.00/1.60k [00:00<04:05, 6.52batch/s]
Training batches on cuda:0:   0% 3.00/1.60k [00:00<02:07, 12.6batch/s]
Training batches on cuda:0:   0% 5.00/1.60k [00:00<01:45, 15.2batch/s]
Training batches on cuda:0:   0% 7.00/1.60k [00:00<01:36, 16.6batch/s]
Training batches on cuda:0:   1% 9.00/1.60k [00:00<01:31, 17.4batch/s]
Training

Streaming output truncated to the last 5000 lines.
Training batches on cuda:0:  37% 600/1.60k [00:31<00:52, 19.2batch/s]
Training batches on cuda:0:  38% 602/1.60k [00:31<00:52, 19.2batch/s]
Training batches on cuda:0:  38% 604/1.60k [00:31<00:52, 19.2batch/s]
Training batches on cuda:0:  38% 606/1.60k [00:31<00:52, 19.2batch/s]
Training batches on cuda:0:  38% 608/1.60k [00:31<00:51, 19.2batch/s]
Training batches on cuda:0:  38% 610/1.60k [00:31<00:51, 19.2batch/s]
Training batches on cuda:0:  38% 612/1.60k [00:31<00:51, 19.2batch/s]
Training batches on cuda:0:  38% 614/1.60k [00:32<00:51, 19.2batch/s]
Training batches on cuda:0:  38% 616/1.60k [00:32<00:51, 19.2batch/s]
Training batches on cuda:0:  39% 618/1.60k [00:32<00:51, 19.2batch/s]
Training batches on cuda:0:  39% 620/1.60k [00:32<00:51, 19.2batch/s]
Training batches on cuda:0:  39% 622/1.60k [00:32<00:51, 19.1batch/s]
Training batches on cuda:0:  39% 624/1.60k [00:32<00:51, 19.2batch/s]
Training batches on cuda:0:  39% 626/1.

## False Edges

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/analysis/assessment/

!python create_false_edges_pykeen.py \
  --dataset_dir /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/graphs/pykeen_factories

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/analysis/assessment
  645 drug nodes available for negative sampling
  458,061 holdout edges across 963 SE types
  C0000731: need 1237 negatives
  C0000737: need 2141 negatives
  C0000768: need 287 negatives
  C0000786: need 149 negatives
  C0000833: need 679 negatives
  C0001122: need 863 negatives
  C0001126: need 93 negatives
  C0001339: need 613 negatives
  C0001418: need 748 negatives
  C0001430: need 202 negatives
  C0001546: need 98 negatives
  C0001623: need 308 negatives
  C0001818: need 116 negatives
  C0001824: need 428 negatives
  C0001948: need 287 negatives
  C0001969: need 149 negatives
  C0002064: need 110 negatives
  C0002103: need 470 negatives
  C0002170: need 844 negatives
  C0002390: need 115 negatives
  C0002453: need 176 negatives
  C0002622: need 1410 negatives
  C0002726: need 84 negatives
  C0002736: need 80 negatives
  C0002792: need 590 negatives
  C0002871: need 2701 negatives
  C0002874: need 167 nega

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/analysis/assessment/
!python assessment_pykeen.py \
        --checkpoint  /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/results/simplE_selfloops/checkpoints/checkpoint_best.pt \
        --dataset_dir /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/graphs/pykeen_factories \
        --out_dir     /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/results/simplE_selfloops/assessment_subset/

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/analysis/assessment
Loading model...
  State-dict checkpoint — rebuilding SimplE  (embedding_dim=256 inferred from 'entity_representations.0._embeddings.weight')
No random seed is specified. This may lead to non-reproducible results.
  Ignored unexpected keys (regularizer state): ['entity_representations.0.regularizer.weight', 'entity_representations.0.regularizer.regularization_term', 'entity_representations.1.regularizer.weight', 'entity_representations.1.regularizer.regularization_term', 'relation_representations.0.regularizer.weight', 'relation_representations.0.regularizer.regularization_term', 'relation_representations.1.regularizer.weight', 'relation_representations.1.regularizer.regularization_term']
  Model type: SimplE
  19,734 entities  |  11,149 relations
Loading holdout edges...
  458,061 holdout edges across 963 SE types

Assessing 963 SE types...
  [1/963] C0000731: AUROC=0.7612  AUPRC=0.6964  AP@50=0.5442
  [2/963]

# Test async checkpointing

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/analysis/assessment/
!python assessment_pykeen_disk_wandb.py \
        --checkpoint  /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/results/simplE_selfloops/checkpoints/checkpoint_best.pt \
        --dataset_dir /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/data/graphs/pykeen_factories \
        --out_dir     /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/results/simplE_selfloops/assessment_subset/

In [3]:
!cat /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/training/train_simple_pykeen_disk_wandb.py

"""
train_simple_pykeen_disk_wandb.py
---------------------------------
Disk-optimised SimplE training (local checkpoints + Drive sync) with optional
Weights & Biases logging. Same training logic as train_simplE_pykeen_disk.py.

Weights & Biases: set --wandb_project (and optionally --wandb_entity,
--wandb_run_name). Use --no_wandb to disable. Requires `pip install wandb` and
`wandb login` (or WANDB_API_KEY).

KEY DIFFERENCE vs train_simplE_pykeen.py
-----------------------------------------
All checkpoint I/O goes to a fast local directory (--local_dir, e.g.
/content/ckpt) instead of Google Drive. This eliminates the ~5 min
per-epoch Drive write overhead that was causing 2.5 h for 4 epochs.

Outputs use fixed filenames (checkpoint_best.pt, training_losses.json).
Use a unique --out_dir / --local_dir per run, or pass --no_clobber on
fresh runs to abort if those paths already contain checkpoints.

Drive sync strategy (three layers):
  1. Periodic sync every --sync_every epochs (default 5)

In [18]:
%cd /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/training
!python train_simple_pykeen_disk_wandb.py \
        --dataset_dir /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/data/graphs/pykeen_factories \
        --out_dir     /content/drive/MyDrive/Colab_Projects_Test/TF_Decagon_Static/results/simplE_selfloops_full \
        --local_dir   /content/ckpt_simplE_async_963 \
        --max_epochs  50 \
        --sync_every 6 \
        --seed        42 \
        --checkpoint_freq 5 \
        --no_tqdm \
        --wandb_project simplE_selfloops_full

/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/training
Device: cuda  |  seed: 42
  Checkpoints (local):  /content/ckpt_simplE_async_963/checkpoints
  Checkpoints (Drive):  /content/drive/MyDrive/Colab_Projects_Test/TF_Decagon_Static/results/simplE_selfloops_full/checkpoints
  Drive sync every:     6 epochs
Loading datasets...
  train: 4,022,003  valid: 502,751  test:  502,751
  entities: 19,734  relations: 11,149
  Entity init: xavier_normal_
No random seed is specified. This may lead to non-reproducible results.
  Parameters: 15,812,096
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your AP

In [ ]:
    !python train_simple_pykeen_disk_wandb.py \\
        --dataset_dir /content/drive/MyDrive/TF_Decagon_Static/data/pykeen/selfloops \\
        --out_dir     /content/drive/MyDrive/TF_Decagon_Static/pykeen_results/simplE_selfloops_fp \\
        --local_dir   /content/ckpt_simplE_fp \\
        --pretrained_entities /content/drive/MyDrive/TF_Decagon_Static/data/embeddings/fp_only/256.npy \\
        --embedding_dim 256 \\
        --wandb_project tf-decagon \\
        --resume      /content/drive/MyDrive/TF_Decagon_Static/pykeen_results/simplE_selfloops_fp/checkpoints/checkpoint_epoch.pt

In [1]:
from google.colab import runtime
runtime.unassign()

## Move directory

In [ ]:
import os

source_path = '/content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/analysis/assessment/false_edges_pykeen'
destination_parent_path = '/content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output'
destination_path = os.path.join(destination_parent_path, 'analysis')

if os.path.exists(source_path):
    print(f'Source directory exists: {source_path}')
    if not os.path.exists(destination_parent_path):
        os.makedirs(destination_parent_path)
        print(f'Created destination parent directory: {destination_parent_path}')

    # Use !mv to move the directory
    !mv "{source_path}" "{destination_parent_path}"
    print(f'Moved {source_path} to {destination_parent_path}')
else:
    print(f'Source directory does not exist: {source_path}')


Source directory exists: /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/results
Moved /content/drive/MyDrive/Colab_Projects/TF_Decagon_Static/results to /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output


In [19]:
import os
import shutil

source_path = '/content/ckpt_simplE_async_963'
destination_root = '/content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/results'
relative_path = ''
destination_parent_path_with_hierarchy = os.path.join(destination_root, relative_path)
destination_final_path = os.path.join(destination_parent_path_with_hierarchy, os.path.basename(source_path))

if os.path.exists(source_path):
    print(f'Source directory exists: {source_path}')

    # Create the full hierarchical destination path if it doesn't exist
    os.makedirs(destination_parent_path_with_hierarchy, exist_ok=True)
    print(f'Created destination hierarchical path: {destination_parent_path_with_hierarchy}')

    # Use shutil.move for Pythonic way or !mv for shell command
    # For this case, !mv is also fine, but needs to specify the full destination.
    !mv "{source_path}" "{destination_parent_path_with_hierarchy}"
    print(f'Moved {os.path.basename(source_path)} to {destination_parent_path_with_hierarchy}')
else:
    print(f'Source directory does not exist: {source_path}')

Source directory exists: /content/ckpt_simplE_async_963
Created destination hierarchical path: /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/results/
Moved ckpt_simplE_async_963 to /content/drive/MyDrive/Colab_Projects_Output/TF_Decagon_Static_Output/results/
